In [ ]:
# Step 0: Prepare Mockup Data
question = "Among the schools with the average score in Math over 560 in the SAT test, how many schools are in the bay area?"
pneuma_retrieved_tables = ["pandas_dfs/codebase_community/comments.csv", "pandas_dfs/codebase_community/posts.csv", "pandas_dfs/california_schools/satscores.csv"]

In [ ]:
import os
from torch import bfloat16

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pipeline_initializer import initialize_pipeline

pipeline = initialize_pipeline(
    'models/qwen25-7b',
    bfloat16,
    8192,
)

In [ ]:
import pandas as pd
from utils import format_schema

schemas_samplerows: dict[str,str] = dict()
for idx, table in enumerate(pneuma_retrieved_tables):
    df = pd.read_csv(f"../TAG-Bench/{table}", nrows=1)
    schemas_samplerows[f"Table_{idx}"] = format_schema(df)

In [ ]:
import pandas as pd
df = pd.read_csv('../TAG-Bench/pandas_dfs/california_schools/satscores.csv')

In [ ]:
df.tail(10)

In [ ]:
schemas_samplerows

In [ ]:
system_prompt = """You are a helpful data scientist who knows specific steps to pre-process a set of tables into a single table that contains only relevant columns & rows to answer a given question.
Even though there are n tables, it is totally fine for a single table to answer the question withou join steps.
It is also fine if some filtering schemes may not be able to be implemented directly. For example, the question may ask about restaurants in the East Coast, but the table only has address information. Just assume that there is a way to filter the rows somehow based on the address information.
Each step is simple enough to be implemented by the users as SQL queries.
Please output your answer as a Python list without any extra formatting.

Example 1:

Question: How many restaurants in the Bay Area that have ratings above 4.0?

Available Tables:
- Table1:
col: rest_id | name | location
sample row: 123 | Burger King | Oakland
- Table2:
col: RestaurantID | Rating
sample row: 88 | 3.1

Your Output: ['Join Table1 and Table2 through `rest_id` and `RestaurantID` columns, resulting in Table3.', 'Select only `name`, `location`, and `Rating` columns from Table3.', 'Filter based on the `Rating` column of Table3 to get restaurants with ratings above 4.0.', 'Filter the rows to get only restaurants in the Bay Area.']

Example 2:

Question: I want to find a school in the 55th street Hyde Park.

Available Tables:
- Table1:
col: id | car wash name | rating
sample row: a134 | Clean Anytime | 4.5
- Table2:
col: id | school | location
sample row: 1 | Payton College Preparatory High School | 1034 N Wells St, Chicago, Illinois

Your Output: ['Select `school` and `location` columns from Table2.', 'Filter rows of Table2 based on the `location` column to get schools in 55th street Hyde Park.']"""

In [ ]:
messages = [
    {
        'role': 'system',
        'content': system_prompt,
    },
    {
        'role': 'user',
        'content': f'Question: {question}\n\nAvailable Tables:'
    },
]
for idx in schemas_samplerows.keys():
    table_content = schemas_samplerows[idx]
    messages[-1]['content'] += f'\n\n{idx}:\n{table_content}'

In [ ]:
from prompting_interface import prompt_pipeline
result = prompt_pipeline(
    pipeline,
    [messages],
    batch_size=1,
    context_length = 8192,
    max_new_tokens=100,
    do_sample=False,
    top_k=None,
    top_p=None,
    penalty_alpha=None,
    temperature=None,
)

In [ ]:
result[0][-1]['content']

## Second Stage

In [ ]:
# import ast
# plans = ast.literal_eval(result[0][-1]['content'])

In [ ]:
plans = ['Select `sname`, `dname`, `AvgScrMath` columns from Table_2.',
 'Filter rows of Table_2 based on the `AvgScrMath` column to get schools with an average score in Math over 560.',
 'Filter the rows to get only schools in the bay area.']

In [ ]:
import os
from torch import bfloat16

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pipeline_initializer import initialize_pipeline

pipeline = initialize_pipeline(
    'models/qwen25-coder-0.5b',
    bfloat16,
    8192,
)

In [ ]:
messages = [
    {
        'role': 'user',
        'content': f'Convert the following statement into SQL query:\n\n{plans[0]}\n\nPlease output the SQL query directly without any formatting.'
    },
]

In [ ]:
from prompting_interface import prompt_pipeline
result = prompt_pipeline(
    pipeline,
    [messages],
    batch_size=1,
    context_length = 8192,
    max_new_tokens=100,
    do_sample=False,
    top_k=None,
    top_p=None,
    penalty_alpha=None,
    temperature=None,
)

In [ ]:
result[0][-1]['content']